In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [9]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="CoRal-project/coral-v2", 
    repo_type="dataset", local_dir="./coral-v2", allow_patterns="read_aloud/*.parquet")

Fetching 310 files:   0%|          | 0/310 [00:00<?, ?it/s]

Fetching 310 files: 100%|██████████| 310/310 [01:29<00:00,  3.47it/s]


'/home/ubuntu/coral-v2'

In [10]:
files = glob('coral-v2/*/*.parquet')
len(files)

310

In [12]:
df = pd.read_parquet(files[0])
df.head()

,id_recording,id_sentence,id_speaker,text,location,location_roomdim,noise_level,noise_type,source_url,age,gender,dialect,country_birth,validated,audio,asr_prediction,asr_validation_model,asr_cer,asr_wer
0,rec_c0931098b5c1c7ce58084dbe600ea1a9,sen_00219948,spe_f9cdb0c4b4c71bc4086c6521a5416c70,De har aldrig deltaget i vinterlege,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Marshall%C3%B8er...,61,female,østjysk,DK,0,{'bytes': b'RIFFFT\x06\x00WAVEfmt \x10\x00\x00...,de har aldrig deltaget i vinterlege,alexandrainst/coral-asr-bootstrap,0.000000,0.166667
1,rec_1f45bddc16abf76942c4f3ca9f347344,sen_00079667,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Siden Gretzky forlod klubben er det ikke lykke...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Los%20Angeles%20...,61,female,østjysk,DK,0,{'bytes': b'RIFF\xc6\xcc\r\x00WAVEfmt \x10\x00...,siden gretzky forlod klubben er det ikke lykke...,alexandrainst/coral-asr-bootstrap,0.000000,0.117647
2,rec_fac2c3e22900009bb55145b21793a2fa,sen_00113990,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Tropperne var også afhængige af de våben som b...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/De%20R%C3%B8de%2...,61,female,østjysk,DK,0,{'bytes': b'RIFF\xc6\x07\x0f\x00WAVEfmt \x10\x...,tropperne var også afhængige af de våben som b...,alexandrainst/coral-asr-bootstrap,0.000000,0.076923
3,rec_87ab8eec45f06c851cebe23694c7d674,sen_00017274,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Især lykkedes det ikke at styrke parlamentets ...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Tyske%20Kejserrige,61,female,østjysk,DK,0,{'bytes': b'RIFFF\x08\x07\x00WAVEfmt \x10\x00\...,især lykkedes det ikke at styrke parlamentets ...,alexandrainst/coral-asr-bootstrap,0.000000,0.125000
4,rec_13cd938cb96f4fdd3a34681712e5d4ae,sen_00073039,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Altools er en række forskellelige værktøjer ud...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Altools,61,female,østjysk,DK,0,{'bytes': b'RIFF\xc6\xbe\x0c\x00WAVEfmt \x10\x...,altools er en række forskellige værktøjer udvi...,alexandrainst/coral-asr-bootstrap,0.028169,0.300000


In [15]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['id_speaker'].iloc[i]}"
            })
        
    return data

In [16]:
# data = loop((files[:1], 0))
# data

In [17]:
data = multiprocessing(files, loop, cores = 40)

100%|██████████| 848/848 [01:13<00:00, 11.52it/s]


In [18]:
with open('coral-v2.json', 'w') as fopen:
    json.dump(data, fopen)

In [19]:
audio_files = [d['audio_filename'] for d in data]

with open('coral-v2-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [20]:
len(data)

261277